In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import joblib

# -------------------------
# Load Diabetes Dataset
# -------------------------
try:
    dataset = pd.read_csv('dataset/diabetes.csv')
    print("Dataset 'diabetes.csv' loaded successfully.")
except FileNotFoundError:
    print("⚠️ File 'dataset/diabetes.csv' not found. Using dummy dataset.")
    data = {
        'Pregnancies': [6, 1, 8, 1, 0, 5, 3, 10, 2, 8],
        'Glucose': [148, 85, 183, 89, 137, 116, 78, 115, 197, 125],
        'BloodPressure': [72, 66, 64, 66, 40, 74, 50, 0, 70, 96],
        'SkinThickness': [35, 29, 0, 23, 35, 0, 32, 0, 45, 0],
        'Insulin': [0, 0, 0, 94, 168, 0, 88, 0, 543, 0],
        'BMI': [33.6, 26.6, 23.3, 28.1, 43.1, 25.6, 31.0, 35.3, 30.5, 0],
        'DiabetesPedigreeFunction': [0.627, 0.351, 0.672, 0.167, 2.288, 0.201, 0.248, 0.134, 0.158, 0.232],
        'Age': [50, 31, 32, 21, 33, 30, 26, 29, 53, 54],
        'Outcome': [1, 0, 1, 0, 1, 0, 1, 0, 1, 1]
    }
    dataset = pd.DataFrame(data)

# -------------------------
# Preprocessing
# -------------------------
numeric_cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']

# Replace 0s with NaN for columns where 0 is invalid
for col in numeric_cols:
    dataset[col] = dataset[col].replace(0, np.nan)
    dataset[col] = dataset[col].fillna(dataset[col].mean())

# Scale numeric features
scaler = StandardScaler()
dataset[numeric_cols] = scaler.fit_transform(dataset[numeric_cols])

# Split features and target
y = dataset['Outcome']
X = dataset.drop(['Outcome'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0)

# -------------------------
# Train Multiple Models
# -------------------------
# KNN
knn_scores = []
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    knn_scores.append(knn.score(X_test, y_test))
best_knn_k = knn_scores.index(max(knn_scores)) + 1

# SVC
svc_scores = []
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
for kernel in kernels:
    svc = SVC(kernel=kernel, probability=True, random_state=0)
    svc.fit(X_train, y_train)
    svc_scores.append(svc.score(X_test, y_test))
best_svc_kernel = kernels[svc_scores.index(max(svc_scores))]

# Decision Tree
dt_scores = []
max_features_range = min(len(X.columns), 20)
for i in range(1, max_features_range + 1):
    dt = DecisionTreeClassifier(max_features=i, random_state=0)
    dt.fit(X_train, y_train)
    dt_scores.append(dt.score(X_test, y_test))
best_dt_features = dt_scores.index(max(dt_scores)) + 1

# Random Forest
rf_scores = []
estimators = [10, 100, 200, 500]
for n in estimators:
    rf = RandomForestClassifier(n_estimators=n, random_state=0)
    rf.fit(X_train, y_train)
    rf_scores.append(rf.score(X_test, y_test))
best_rf_n = estimators[rf_scores.index(max(rf_scores))]

# -------------------------
# Select Best Model
# -------------------------
models = {
    "KNN": max(knn_scores),
    "SVC": max(svc_scores),
    "DecisionTree": max(dt_scores),
    "RandomForest": max(rf_scores)
}
best_model_name = max(models, key=models.get)
print(f"\nBest model selected: {best_model_name}")

if best_model_name == "KNN":
    best_model = KNeighborsClassifier(n_neighbors=best_knn_k)
elif best_model_name == "SVC":
    best_model = SVC(kernel=best_svc_kernel, probability=True, random_state=0)
elif best_model_name == "DecisionTree":
    best_model = DecisionTreeClassifier(max_features=best_dt_features, random_state=0)
else:
    best_model = RandomForestClassifier(n_estimators=best_rf_n, random_state=0)

best_model.fit(X_train, y_train)

# -------------------------
# Evaluate
# -------------------------
y_pred = best_model.predict(X_test)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -------------------------
# Save model, scaler, and feature columns using joblib
# -------------------------
joblib.dump(best_model, 'diabetes_model.pkl')
joblib.dump(scaler, 'diabetes_scaler.pkl')
joblib.dump(X_train.columns.tolist(), 'diabetes_features.pkl')
print("\n✅ Model, scaler, and feature columns saved with joblib.")

# -------------------------
# Predict New Patient
# -------------------------
def predict_diabetes(new_data):
    df = pd.DataFrame([new_data])
    scaler_loaded = joblib.load('diabetes_scaler.pkl')
    features = joblib.load('diabetes_features.pkl')
    model_loaded = joblib.load('diabetes_model.pkl')
    
    df[numeric_cols] = scaler_loaded.transform(df[numeric_cols])
    
    # Ensure same column order
    df = df[features]
    
    proba = model_loaded.predict_proba(df)[0][1]
    prob_percentage = proba * 100
    
    if prob_percentage >= 70:
        chance = "HIGH CHANCE"
    elif prob_percentage >= 30:
        chance = "MEDIUM CHANCE"
    else:
        chance = "LOW CHANCE"
    
    print(f"\n🎯 Diabetes Probability: {prob_percentage:.2f}% | Confidence: {chance}")

# -------------------------
# Example High Chance Input
# -------------------------
high_chance_patient = {
    'Pregnancies': 6,                
    'Glucose': 180,                  
    'BloodPressure': 90,             
    'SkinThickness': 35,             
    'Insulin': 200,                  
    'BMI': 35.0,                     
    'DiabetesPedigreeFunction': 0.8, 
    'Age': 50                         
}

# -------------------------
# Example Low Chance Input
# -------------------------
low_chance_patient = {
    'Pregnancies': 0,                
    'Glucose': 90,                   
    'BloodPressure': 70,             
    'SkinThickness': 20,             
    'Insulin': 80,                   
    'BMI': 22.0,                     
    'DiabetesPedigreeFunction': 0.1, 
    'Age': 25                         
}

# Predict
predict_diabetes(high_chance_patient)
predict_diabetes(low_chance_patient)


Dataset 'diabetes.csv' loaded successfully.

Best model selected: KNN

Confusion Matrix:
 [[156  14]
 [ 42  42]]

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.92      0.85       170
           1       0.75      0.50      0.60        84

    accuracy                           0.78       254
   macro avg       0.77      0.71      0.72       254
weighted avg       0.78      0.78      0.77       254


✅ Model, scaler, and feature columns saved with joblib.

🎯 Diabetes Probability: 100.00% | Confidence: HIGH CHANCE

🎯 Diabetes Probability: 0.00% | Confidence: LOW CHANCE
